<a href="https://colab.research.google.com/github/ecuirty-kr/1_DataAnalysis/blob/main/p10_type2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[구글 코랩(Colab)에서 실행하기](https://colab.research.google.com/github/lovedlim/bae-v3/blob/main/part4/ch10/p10_type2.ipynb)

In [14]:
## 문제 정의 : 제공된 학습용 데이터(train.csv)는 서울시 각 시군구별 건물 및 시설 현황 데이터이다. 학습용 데이터를 활용하여 총가스사용량을
## 예측하는 모형을 개발하고, 모형을 평가용 데이터(test.csv)에 적용하여 총가스사용량을 예측하시오.
## 예측 결과는 RMSE(Root Mean Squared Error) 평가 지표에 따라 평가함
## 타겟 변수(총가스사용량)는 일부 값이 0으로 기재되어 있으며, 이는 결측치를 대체한 값

## 제출 파일명 : result.csv
## 예측 총가스사용량 컬럼명 : pred
## 제출 컬럼 개수 : pred 컬럼 1개
## 평가용 데이터 개수와 예측 결과 데이터 개수 일치 : 1,476개

# 1. 라이브러리 및 데이터
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_test.csv")
# 데이터 샘플 확인
# print(train.head(5),"\n",test.head(5))

# 2. 탐색적 데이터 분석 (크기 / 자료형 / 결측치 / target 빈도 수)
# print(train.shape, "\n", test.shape)  # train (3196, 6)  / test  (1476, 5)
### 에어컨을 계속 틀면 춥고 끄면 열기 때문에 덥고-..
# print(train.info(), test.info()) ## object 1 / int 4 (train float 1 추가) : 혼합 데이터 -> 인코딩 필요
# print(train.isnull().sum(), test.isnull().sum())  # 결측치 없음
print(train['총가스사용량'].value_counts()) ## 0.0 이 대다수

# 3. 데이터 전처리 - 인코딩 : 범주형 -> 수치형 변환으로 원-핫 인코딩 수행할 예정
target = train.pop('총가스사용량')

# from sklearn. ## 오 인코딩 라이브러리 까먹음
## 까먹은게 아니라 없네.

# 4. 원-핫 인코딩
train = pd.get_dummies(train)
test = pd.get_dummies(test)  ## 인코딩 후 데이터 크기 재확인 권장
print(train.shape, test.shape)  # (3196, 25) (1476, 25)

# 5. 검증 데이터 분리
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# 6. 머신러닝 학습 및 평가
from sklearn.ensemble import RandomForestRegressor ## 구할 가스사용량은 연속된 수치값이므로 회귀 모델 사용
rf = RandomForestRegressor(random_state=0)
rf.fit(X_train, y_train)
pred = rf.predict(X_val) ## predict()에는 X_val 임

# 평가지표 : RMSE
from sklearn.metrics import root_mean_squared_error
rmse = root_mean_squared_error(y_val, pred)
print("RF rmse : ", rmse) # 결과 :  960.485846380754

# 6-1. LightGBM 학습 및 평가
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_train, y_train)
pred = lg.predict(X_val)

result = root_mean_squared_error(y_val, pred)
print("LG rmse : ", result)  # 결과 : 1064.8095758723994

# RF RMSE < LGB RMSE  이므로 RF 가 우수한 성능 : 회귀 모델 평가지표는 값이 "작을수록" 우수함

# 7. 결과 예측 및 파일 생성
pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)

## 검증
print(pred.shape) # (1476,) 1,476 행으로 구성
file = pd.read_csv("result.csv")
file

총가스사용량
0.0        57
10693.8     3
11144.2     3
12011.1     3
10203.3     3
           ..
9237.7      1
10233.3     1
9209.0      1
10769.2     1
10275.3     1
Name: count, Length: 3015, dtype: int64
(3196, 25) (1476, 25)
RF rmse :  960.485846380754
LG rmse :  1064.8095758723994
(1476,)


,pred
0,9787.863633
1,10770.220091
2,9137.436055
3,9787.863633
4,11234.287608
...,...
1471,12258.710856
1472,10904.053927
1473,9895.018350
1474,9827.441850


In [ ]:
# 문제정의
# 평가: RMSE
# target: 총가스사용량
# 최종파일: result.csv(컬럼 1개 pred)

# 라이브러리 및 데이터 불러오기
import pandas as pd
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_test.csv")

# 탐색적 데이터 분석(EDA)
print("===== 데이터 크기 =====")
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)

print("\n ===== 데이터 샘플 =====")
print(train.head())

print("\n ===== 데이터 정보(자료형) =====")
print(train.info())

print("\n ===== train object컬럼 unique 수 =====")
print(train.describe(include='object'))

print("\n ===== test object컬럼 unique 수 =====")
print(test.describe(include='object'))

print("\n ===== train 결측치 수 =====")
print(train.isnull().sum().sum())

print("\n ===== test 결측치 수 =====")
print(test.isnull().sum().sum())

print("\n ===== target 기술 통계 =====")
print(train['총가스사용량'].describe())

===== 데이터 크기 =====
Train Shape: (3196, 6)
Test Shape: (1476, 5)

 ===== 데이터 샘플 =====
  시군구명  생활및판매  공공문화  복지의료  업무오락체육   총가스사용량
0  구로구      2     0     0       0   9077.8
1  구로구      6     0     1       2  10105.5
2  구로구     27     0     0       0   8603.6
3  구로구      2     0     0       0  11076.8
4  구로구     83     0     1      19  10781.4

 ===== 데이터 정보(자료형) =====
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3196 entries, 0 to 3195
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   시군구명    3196 non-null   object 
 1   생활및판매   3196 non-null   int64  
 2   공공문화    3196 non-null   int64  
 3   복지의료    3196 non-null   int64  
 4   업무오락체육  3196 non-null   int64  
 5   총가스사용량  3196 non-null   float64
dtypes: float64(1), int64(4), object(1)
memory usage: 149.9+ KB
None

 ===== train object컬럼 unique 수 =====
        시군구명
count   3196
unique    21
top      강남구
freq     715

 ===== test object컬럼 unique 수 =====
        시군구명
count  

In [ ]:
# 데이터
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_test.csv")


# 데이터 전처리
target = train.pop('총가스사용량')

# 원핫 인코딩
# print("인코딩 전:", train.shape, test.shape)
train = pd.get_dummies(train)
test = pd.get_dummies(test)
# print("인코딩 후:", train.shape, test.shape)


# 검증데이터 분리
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# RMSE (Root Mean Squared Error)
from sklearn.metrics import root_mean_squared_error

# 랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)
print("랜덤포레스트:",root_mean_squared_error(y_val, pred))

# LightGBM
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)
print("lightgbm:",root_mean_squared_error(y_val, pred))

# 최종 제출 파일
pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)

랜덤포레스트: 960.485846380754
lightgbm: 1064.8095758723994


In [ ]:
# 1. pred 행의 수와 test의 행의 수 비교
print("pred: ",pred.shape) # test 행의 수: 1476

# 2. 생성한 csv 확인
print(pd.read_csv("result.csv").head())

pred:  (1476,)
           pred
0   9787.863633
1  10770.220091
2   9137.436055
3   9787.863633
4  11234.287608


[심화] 결측치(타겟: 0)을 제거한다면?

In [ ]:
# 데이터
train = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_train.csv")
test = pd.read_csv("https://raw.githubusercontent.com/lovedlim/bae-v3/main/part4/ch10/gas_test.csv")

# 데이터 전처리
print("제거 전:", train.shape)
cond = train['총가스사용량'] == 0
train = train[~cond]
print("제거 후:", train.shape)

target = train.pop('총가스사용량')

# 원핫 인코딩
# print("인코딩 전:", train.shape, test.shape)
train = pd.get_dummies(train)
test = pd.get_dummies(test)
# print("인코딩 후:", train.shape, test.shape)


# 검증데이터 분리
from sklearn.model_selection import train_test_split
X_tr, X_val, y_tr, y_val = train_test_split(train, target, test_size=0.2, random_state=0)

# RMSE (Root Mean Squared Error)
from sklearn.metrics import root_mean_squared_error

# 랜덤포레스트
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state=0)
rf.fit(X_tr, y_tr)
pred = rf.predict(X_val)
print("랜덤포레스트:",root_mean_squared_error(y_val, pred))

# LightGBM
import lightgbm as lgb
lg = lgb.LGBMRegressor(random_state=0, verbose=-1)
lg.fit(X_tr, y_tr)
pred = lg.predict(X_val)
print("lightgbm:",root_mean_squared_error(y_val, pred))

# 최종 제출 파일
pred = rf.predict(test)
result = pd.DataFrame({'pred':pred})
result.to_csv("result.csv", index=False)

제거 전: (3196, 6)
제거 후: (3139, 6)
랜덤포레스트: 673.8026644431536
lightgbm: 741.9117749289708
